# Wildfire Event EDA (Synthetic)

This notebook demonstrates the exploratory steps used to validate the ingestion pipeline on a small synthetic dataset.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from shapely.geometry import Point
from shapely.affinity import scale

rng = np.random.default_rng(42)
events = []
for idx in range(5):
    center = Point(-121 + rng.normal(scale=0.5), 38 + rng.normal(scale=0.5))
    polygon = scale(center.buffer(0.2 + rng.random() * 0.15), xfact=1, yfact=0.6)
    events.append(
        {
            "event_id": f"EDA_{idx:02d}",
            "area_sq_km": polygon.area * 12_000,
            "centroid_lon": center.x,
            "centroid_lat": center.y,
        }
    )
df = pd.DataFrame(events)
df

## Area distribution

In [ ]:
df[['event_id', 'area_sq_km']].describe()

## Save sample GeoJSON

In [ ]:
out = [
    {
        "type": "Feature",
        "properties": {"event_id": row.event_id},
        "geometry": Point(row.centroid_lon, row.centroid_lat).buffer(0.2).__geo_interface__,
    }
    for row in df.itertuples()
]
geojson = {"type": "FeatureCollection", "features": out}
Path('eda_events.geojson').write_text(json.dumps(geojson, indent=2))
geojson['features'][0]